# 🔄 Checkpoint & Session Recovery Setup

**NEW: Füge diese Zelle NACH Step 9 (Configure Training Parameters) ein**

Diese Zelle sorgt dafür, dass:
- ✅ Alle Checkpoints auf Google Drive gespeichert werden
- ✅ Training nach Session-Disconnect automatisch fortsetzt
- ✅ Fortschritt nie verloren geht


In [ ]:
# ==========================================
# CHECKPOINT & SESSION RECOVERY SETUP
# ==========================================

import glob
import json
from datetime import datetime
from pathlib import Path

print("🔄 Setting up checkpoint persistence and session recovery...\n")

# Create checkpoint directory on Google Drive (persistent across sessions)
CHECKPOINT_DIR = Path("/content/drive/MyDrive/piper_training/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Session state file to track progress
SESSION_STATE_FILE = Path("/content/drive/MyDrive/piper_training/session_state.json")

# Also copy checkpoints to a backup location
BACKUP_CHECKPOINT_DIR = Path("/content/drive/MyDrive/piper_training/checkpoints_backup")
BACKUP_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Check for existing checkpoints to resume from
existing_checkpoints = sorted(glob.glob(str(CHECKPOINT_DIR / "*.ckpt")))
backup_checkpoints = sorted(glob.glob(str(BACKUP_CHECKPOINT_DIR / "*.ckpt")))

# Determine best checkpoint to resume from
all_checkpoints = existing_checkpoints + backup_checkpoints
all_checkpoints = sorted(all_checkpoints, key=lambda x: Path(x).stat().st_mtime)

if all_checkpoints:
    RESUME_CHECKPOINT = all_checkpoints[-1]  # Most recent
    
    # Load session state if exists
    if SESSION_STATE_FILE.exists():
        with open(SESSION_STATE_FILE, 'r') as f:
            session_state = json.load(f)
        
        print("🔄 PREVIOUS TRAINING SESSION FOUND!")
        print("=" * 50)
        print(f"   Voice name: {session_state.get('voice_name', 'Unknown')}")
        print(f"   Language: {session_state.get('language', 'Unknown')}")
        print(f"   Started: {session_state.get('started', 'Unknown')}")
        if 'last_checkpoint_time' in session_state:
            print(f"   Last checkpoint: {session_state['last_checkpoint_time']}")
        print(f"   Total checkpoints: {len(all_checkpoints)}")
        print(f"\n   📁 Resume from: {Path(RESUME_CHECKPOINT).name}")
        print("\n   ✅ Training will continue from this checkpoint!")
        print("=" * 50)
    else:
        print(f"🔄 Found {len(all_checkpoints)} checkpoint(s)")
        print(f"   Will resume from: {Path(RESUME_CHECKPOINT).name}\n")
else:
    RESUME_CHECKPOINT = CKPT_URL if CKPT_URL else None
    if RESUME_CHECKPOINT:
        print("▶️ NEW TRAINING SESSION")
        print(f"   Starting from base checkpoint\n")
    else:
        print("▶️ NEW TRAINING SESSION")
        print(f"   Starting from scratch (no checkpoint)\n")

# Save/update session state
session_state = {
    'voice_name': VOICE_NAME,
    'language': ESPEAK_VOICE,
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'backup_dir': str(BACKUP_CHECKPOINT_DIR),
    'started': datetime.now().isoformat(),
    'resume_from': RESUME_CHECKPOINT,
    'sample_rate': SAMPLE_RATE_HZ,
    'batch_size': BATCH_SIZE
}

with open(SESSION_STATE_FILE, 'w') as f:
    json.dump(session_state, f, indent=2)

print(f"💾 Session state saved to Google Drive")
print(f"   State file: {SESSION_STATE_FILE}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")
print(f"   Backup: {BACKUP_CHECKPOINT_DIR}")
print(f"\n⚡ If your session disconnects:")
print(f"   1. Simply re-run all cells from the beginning")
print(f"   2. Training will automatically resume")
print(f"   3. No progress will be lost!\n")

# 🚀 Improved Training Cell

**ERSETZE die bestehende Training-Zelle (Step 12) mit dieser:**

Verbesserungen:
- ✅ Speichert Checkpoints auf Google Drive
- ✅ Erstellt automatisch Backups
- ✅ Tracked Training-Progress
- ✅ Ermöglicht automatisches Resume

In [ ]:
# ==========================================
# START TRAINING WITH CHECKPOINT PERSISTENCE
# ==========================================

import shutil
import time

print("\n" + "="*60)
print("🚀 STARTING PIPER TRAINING")
print("="*60)
print("   ⏱️  Duration: 2-12+ hours (depending on checkpoint availability)")
print("   💾  Checkpoints: Auto-saved to Google Drive every epoch")
print("   🔄  Resume: Automatic if session disconnects")
print("   💡  Tip: You can close this tab - training continues")
print("="*60 + "\n")

# Lightning logs directory on Google Drive for full persistence
LIGHTNING_LOGS_DIR = Path("/content/drive/MyDrive/piper_training/lightning_logs")
LIGHTNING_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Build training command with Google Drive paths
training_command = f"""python3 -m piper.train fit \\
  --data.voice_name "{VOICE_NAME}" \\
  --data.csv_path "{str(METADATA_CSV)}" \\
  --data.audio_dir "{str(AUDIO_DIR)}" \\
  --model.sample_rate {SAMPLE_RATE_HZ} \\
  --data.espeak_voice "{ESPEAK_VOICE}" \\
  --data.cache_dir "{str(CACHE_DIR)}" \\
  --data.config_path "{str(CONFIG_PATH)}" \\
  --data.batch_size {BATCH_SIZE} \\
  --trainer.default_root_dir "{str(LIGHTNING_LOGS_DIR)}" \\
  --trainer.enable_checkpointing true \\
  --trainer.max_epochs 10000"""

# Add checkpoint path for resume
if RESUME_CHECKPOINT:
    training_command += f" \\
  --ckpt_path "{RESUME_CHECKPOINT}""
    print(f"📂 Resuming from checkpoint: {Path(RESUME_CHECKPOINT).name}\n")
else:
    print(f"📂 Starting fresh training\n")

# Update session state before training
session_state['training_started'] = datetime.now().isoformat()
session_state['training_command'] = training_command
with open(SESSION_STATE_FILE, 'w') as f:
    json.dump(session_state, f, indent=2)

print("📊 Starting training now...\n")
start_time = time.time()

# Execute training
!{training_command}

elapsed_time = time.time() - start_time
hours, remainder = divmod(elapsed_time, 3600)
minutes, seconds = divmod(remainder, 60)

print("\n" + "="*60)
print("✅ TRAINING COMPLETED!")
print("="*60)
print(f"   ⏱️  Duration: {int(hours)}h {int(minutes)}m {int(seconds)}s")

# Find all generated checkpoints
final_checkpoints = sorted(glob.glob(str(LIGHTNING_LOGS_DIR / "**/*.ckpt"), recursive=True))

if final_checkpoints:
    print(f"   📦  Generated {len(final_checkpoints)} checkpoint(s)")
    
    # Copy latest checkpoint to main checkpoint directory
    latest_ckpt = final_checkpoints[-1]
    dest_ckpt = CHECKPOINT_DIR / Path(latest_ckpt).name
    shutil.copy2(latest_ckpt, dest_ckpt)
    
    # Also create backup
    backup_ckpt = BACKUP_CHECKPOINT_DIR / f"{VOICE_NAME}_final_{datetime.now().strftime('%Y%m%d_%H%M%S')}.ckpt"
    shutil.copy2(latest_ckpt, backup_ckpt)
    
    print(f"   💾  Main checkpoint: {dest_ckpt}")
    print(f"   💾  Backup: {backup_ckpt}")
    
    # Update session state
    session_state['training_completed'] = datetime.now().isoformat()
    session_state['final_checkpoint'] = str(dest_ckpt)
    session_state['backup_checkpoint'] = str(backup_ckpt)
    session_state['training_duration_seconds'] = elapsed_time
    session_state['total_checkpoints'] = len(final_checkpoints)
    
    with open(SESSION_STATE_FILE, 'w') as f:
        json.dump(session_state, f, indent=2)
    
    print(f"\n   📁 All training data saved to Google Drive")
    print(f"   📄 Session state: {SESSION_STATE_FILE}")
else:
    print(f"   ⚠️  No checkpoints found - training may have failed")

print("="*60)

# 📊 Training Progress Monitor

**NEUE ZELLE: Füge diese ein, um den Fortschritt zu überwachen**

In [ ]:
# ==========================================
# MONITOR TRAINING PROGRESS
# ==========================================

import json
import glob
from pathlib import Path
from datetime import datetime

print("📊 Training Progress Report")
print("="*60)

# Load session state
SESSION_STATE_FILE = Path("/content/drive/MyDrive/piper_training/session_state.json")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/piper_training/checkpoints")
LIGHTNING_LOGS_DIR = Path("/content/drive/MyDrive/piper_training/lightning_logs")

if SESSION_STATE_FILE.exists():
    with open(SESSION_STATE_FILE, 'r') as f:
        state = json.load(f)
    
    print(f"Voice Name: {state.get('voice_name', 'N/A')}")
    print(f"Language: {state.get('language', 'N/A')}")
    print(f"Started: {state.get('started', 'N/A')}")
    
    if 'training_started' in state:
        print(f"Training Started: {state['training_started']}")
    
    if 'training_completed' in state:
        print(f"Training Completed: {state['training_completed']}")
        duration = state.get('training_duration_seconds', 0)
        hours, remainder = divmod(duration, 3600)
        minutes, seconds = divmod(remainder, 60)
        print(f"Duration: {int(hours)}h {int(minutes)}m {int(seconds)}s")
    
    print(f"\nCheckpoint Info:")
    
    # Count checkpoints
    main_ckpts = list(CHECKPOINT_DIR.glob("*.ckpt"))
    lightning_ckpts = list(LIGHTNING_LOGS_DIR.glob("**/*.ckpt"))
    
    print(f"  Main checkpoints: {len(main_ckpts)}")
    print(f"  Lightning logs checkpoints: {len(lightning_ckpts)}")
    
    if 'final_checkpoint' in state:
        print(f"  Final checkpoint: {Path(state['final_checkpoint']).name}")
    
    if 'backup_checkpoint' in state:
        print(f"  Backup checkpoint: {Path(state['backup_checkpoint']).name}")
    
    # List recent checkpoints
    if main_ckpts:
        print(f"\n📁 Recent Checkpoints:")
        for ckpt in sorted(main_ckpts, key=lambda x: x.stat().st_mtime, reverse=True)[:5]:
            size_mb = ckpt.stat().st_size / (1024 * 1024)
            mtime = datetime.fromtimestamp(ckpt.stat().st_mtime)
            print(f"  - {ckpt.name} ({size_mb:.1f} MB) - {mtime.strftime('%Y-%m-%d %H:%M')}")
else:
    print("No session state found. Run the training cells first.")

print("="*60)